# 02 · Feature engineering

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| **2. Feature engineering** | **this notebook** |
| 3. Model training | `03_model_training.ipynb` |
| 4. Evaluation | `04_evaluation.ipynb` |

Everything here calls `src/aerocast/features.py`, `data.py` and `leakage.py`, the same code the pipeline script uses.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display
from aerocast import config, data, eda, features, leakage, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 160)
R = config.RESULTS_DIR

## 1. The decision clock

> Predictions for all flights on day **D** are issued at **20:00 NYC time on D-1**. An outcome may feed a feature for D **only if it was known by then**.

| Information | Allowed? | When it becomes known |
|---|---|---|
| Timetable for day D (time, carrier, route, distance, aircraft) | ✅ | published in advance |
| Flights scheduled at the same airport/hour on D | ✅ | timetable, **including flights later cancelled** |
| Arrival delay of an earlier flight | ✅ once known | at its **actual gate arrival**, on the NYC clock |
| Cancellation of an earlier flight | ✅ once known | at its scheduled departure (if cancelled) or actual departure |
| Anything that happens on day D | ❌ | after the decision |

**Implementation.** `data.add_target` gives each flight `arr_known_dt` and `dep_known_dt`. `data.available_date` maps a known-time *t* to the first target date whose decision time is at or after *t*: `ceil_day(t − 20h) + 1 day`. Then `features.rolling_past` sums outcomes over trailing windows of these **availability dates**, not flight dates.

### 1.1 Arrival time on the NYC clock

`arr_time` is in the **destination's** local time, and the dataset has no time-zone table. The offset is recoverable, though: (local arrival − local departure) − air time = taxi time + whole-hour offset, and taxi time (about 25 min) is much smaller than an hour. The offsets are stored as a **fixed lookup table** in `config.DEST_OFFSET_MIN`, together with the 2013 daylight-saving rule (HNL, PHX, Puerto Rico and the US Virgin Islands don't observe DST). Time zones are static geography, so they must not be re-estimated from the flights being featurised, or past features would depend on future data.

In [2]:
df = data.add_target(data.load_raw())
derived = data.dest_offsets(df)
print("table reproduced from data:", all(config.DEST_OFFSET_MIN[d] == o for d, o in derived.items()))
pd.Series(config.DEST_OFFSET_MIN).value_counts().sort_index().rename("destinations").to_frame().T.rename(columns=lambda o: f"{o:+d} min")

table reproduced from data: True


,-360 min,-240 min,-180 min,-120 min,-60 min,+0 min
destinations,1,1,14,8,21,60


### 1.2 How much of yesterday is known at 20:00?

In [3]:
eda.arrival_timing_summary(df)

,flights,delay_rate,share
usable_after_days,,,
1,225400,0.1829,0.6886
2,101946,0.3814,0.3114


About **31% of recorded arrival outcomes become available too late for the next day's 20:00 D-1 decision**, and their delay rate is about 38%, versus 18% for outcomes available earlier. This motivates checking availability rather than relying on flight date alone.

The 20:00 clock scores 0.4715, compared with 0.4589 at 00:00 (approximately a two-day lag) and 0.4814 at 24:00. The 24:00 scenario permits outcomes unavailable at the chosen 20:00 deadline. These scenarios move fitting and selection cutoffs as well as feature-history cutoffs, so their score differences do not isolate a feature-leakage effect.

The earlier midnight-history implementation admitted outcomes unknown at the chosen 20:00 prediction deadline. The configured clock excludes them. The score comparisons above are clock-sensitivity checks, not isolated estimates of how much feature leakage increased AP.

## 2. Build all features

Features are built on the **full** schedule (including cancelled flights). Modelling rows are filtered afterwards.

In [4]:
X, groups = features.build_features(df)
print(X.shape, "| smoothing m =", config.SMOOTHING_M, "| long window =", config.LONG_WINDOW, "days")
pd.Series({g: len(c) for g, c in groups.items()}, name="n_features").to_frame()

(336776, 45) | smoothing m = 5 | long window = 14 days


,n_features
categorical,3
schedule,10
congestion,7
history_entity,18
history_day,7


In [5]:
train = (df["date"] > config.WARMUP_END) & (df["date"] <= config.SPLITS["train"][1]) & df["is_delayed"].notna()
print(f"training rows: {train.sum():,}")

training rows: 211,684


## 3. Feature groups

### 3.1 Identifiers (`categorical`)
`carrier`, `origin`, `dest` as categoricals. XGBoost splits on them natively. LogReg one-hot encodes them and groups categories with fewer than 50 flights.

In [6]:
X.loc[train, groups["categorical"]].describe()

,carrier,origin,dest
count,211684,211684,211684
unique,16,3,102
top,UA,EWR,ATL
freq,37403,76559,10924


### 3.2 Schedule & calendar (`schedule`)
* **Cyclic encodings** of scheduled departure time, scheduled arrival time and day of week: sin/cos of 2π·value/period, so 23:59 and 00:01 are neighbours.
* `dep_hour`, `is_weekend`, `days_to_holiday` (US **federal** holiday calendar, capped at 14), and `log_distance`.
* No month feature (§7).

In [7]:
fig = plots.feature_histograms(X.loc[train], groups["schedule"], "Schedule & calendar features (train)")
plots.save(fig, "feat_schedule_dist"); fig

<Figure size 1240x630 with 12 Axes>

### 3.3 Scheduled congestion & aircraft rotation (`congestion`)
Counts over the **published timetable** of day D: flights in the same origin-hour, the ±1-hour bank, the day; the destination's daily volume; the carrier's volume at the origin. `tail_leg_idx` / `tail_legs_day` give the aircraft's NYC departure number that day. This assumes the **aircraft (tail number) assignment is known the day before**.

In [8]:
fig = plots.feature_histograms(X.loc[train], groups["congestion"], "Congestion & rotation features (train)")
plots.save(fig, "feat_congestion_dist"); fig

<Figure size 1240x420 with 8 Axes>

### 3.4 Entity-level history (`history_entity`)
Smoothed delay rates plus log volume for carrier (7 d, 14 d), destination, route, carrier-route, origin × hour, flight number and tail number (14 d). The log volume (`hist_logn_*`) tells the model how much evidence each rate rests on.

In [9]:
fig = plots.feature_histograms(X.loc[train], groups["history_entity"], "Entity-level history features (train)")
plots.save(fig, "feat_history_entity_dist"); fig

<Figure size 1240x1050 with 20 Axes>

### 3.5 Day-level history (`history_day`)
Network-wide and per-origin delay rate over the last 24 hours known before the decision (`*_1d`), the 7-day network rate, the cancellation rates, and the mean arrival delay at the origin.

In [10]:
fig = plots.feature_histograms(X.loc[train], groups["history_day"], "Day-level history features (train)", ncols=4)
plots.save(fig, "feat_history_day_dist"); fig

<Figure size 1240x420 with 8 Axes>

## 4. Worked example: decision clock + smoothing

`features.history_trace` reruns the **real** `rolling_past` and `smoothed_rate` for one key and reports them day by day. Example: route **EWR → CAE**, a thin route with one evening flight every 3–4 days, using the 14-day window.

The rate is a beta-binomial (Bayesian) shrinkage estimate:

$$\hat r_{k,D} = \frac{\text{late}_k + m\,\pi_D}{\text{flights}_k + m}, \qquad m = 5$$

where the counts cover outcomes **available** for D within the window, and $\pi_D$ is the network delay rate over the 28 days before D (past-only).

In [11]:
trace = features.history_trace(df, ("origin", "dest"), ("EWR", "CAE"), window=config.LONG_WINDOW)
trace.head(12)

,flights_today,late_today,arrival_nyc,outcome_usable_from,hist_flights,hist_late,network_prior,hist_rate
date,,,,,,,,
2013-01-03,1,1.0,2013-01-03 21:02:00,2013-01-05,0.0,0.0,0.2982,0.2982
2013-01-06,1,1.0,2013-01-06 21:18:00,2013-01-08,1.0,1.0,0.2469,0.3724
2013-01-10,1,0.0,2013-01-10 20:31:00,2013-01-12,2.0,2.0,0.2017,0.4298
2013-01-13,1,1.0,2013-01-13 22:31:00,2013-01-15,3.0,2.0,0.1794,0.3621
2013-01-17,1,1.0,2013-01-17 22:27:00,2013-01-19,4.0,3.0,0.2058,0.4477
2013-01-20,1,1.0,2013-01-20 20:51:00,2013-01-22,4.0,3.0,0.2065,0.4480
2013-01-24,1,1.0,2013-01-24 22:10:00,2013-01-26,4.0,3.0,0.2108,0.4505
2013-01-27,1,0.0,2013-01-27 20:25:00,2013-01-29,4.0,4.0,0.2244,0.5691
2013-01-31,0,0.0,NaT,NaT,4.0,3.0,0.2185,0.4547


How to read it:
* **The decision clock in action.** Every flight on this route lands after 20:00 NYC (e.g. 3 Jan at 21:02), so `outcome_usable_from` is **two** days later (5 Jan). That outcome isn't known at 20:00 on 3 Jan, so it can't inform 4 Jan.
* **6 Jan**: the history holds exactly one flight (3 Jan, late). The rate is $(1 + 5 \times 0.2469)/(1+5) = 0.372$, not 1.0. The shrinkage pulls a 1-out-of-1 record toward the network level.
* **20 Jan**: the 14-day window covers availability dates 7–20 Jan, i.e. the flights of 6, 10, 13 and 17 Jan (3 of 4 late). The 3 Jan flight (available 5 Jan) has dropped out.
* **31 Jan**: `flights_today = 0`. The flight was cancelled, so it's on the schedule but has no arrival outcome.
* `m = 5` and the 14-day window were chosen by rolling CV (§6).

## 5. Leakage checks (inline)

Both checks live in `aerocast.leakage` and also run in `tests/test_no_leakage.py`.

**(a) Future outcomes.** Overwrite every post-departure column from a cutoff date onward with random values, rebuild all features, and require features up to and including the cutoff to be **bit-for-bit unchanged**. Features after the cutoff must change, otherwise the check would be vacuous.

In [12]:
check = leakage.future_dependence_check(df, "2013-06-15", X=X)
assert check["changed_up_to_cutoff"] == 0 and check["changed_after_cutoff"] > 0
pd.Series({k: v for k, v in check.items() if k != "changed_columns_after"}).to_frame("value")

,value
cutoff,2013-06-15
rows_up_to_cutoff,151859
changed_up_to_cutoff,0
rows_after_cutoff,184917
changed_after_cutoff,4558344


**(b) The decision clock.** For a target day D: randomise the arrival outcomes of D-1 flights that landed **after** 20:00 on D-1 (timing columns untouched), and require every feature of D to be unchanged. As a positive control, randomising D-1 flights that landed **before** 20:00 must change D's history features.

In [13]:
rows = [leakage.decision_clock_check(df, d, X=X) for d in ["2013-02-14", "2013-07-11", "2013-12-20"]]
clock = pd.DataFrame(rows).set_index("target_date")
assert (clock["changed_cells_on_D_after_scrambling_late"] == 0).all()
assert (clock["changed_cells_on_D_after_scrambling_early"] > 0).all()
clock

,decision_time,d_minus_1_flights_after_clock,d_minus_1_flights_before_clock,changed_cells_on_D_after_scrambling_late,changed_cells_on_D_after_scrambling_early
target_date,,,,,
2013-02-14,2013-02-13 20:00:00,267,638,0,12428
2013-07-11,2013-07-10 20:00:00,292,558,0,13078
2013-12-20,2013-12-19 20:00:00,322,647,0,12740


**Both pass.** Late-evening arrivals have zero influence on the next day's features, while daytime arrivals do influence them, so the check isn't vacuous.

## 6. Choosing smoothing strength m, the long window, C, and the clock (rolling CV)

The first version used hand-picked m = 20 and a 28-day long window. Each pair from {5, 20, 50} × {14, 28, 56} is now scored by rolling-origin CV inside the training window, at its best C from the Logistic Regression grid. The `cv_pr_auc_mean_C0.001` column retains the fixed-C comparison.

The final two rows repeat m = 5 and a 14-day window at 00:00 and 24:00 on D-1. Moving the clock changes outcome availability for feature history, model fitting and selection. Validation scores exclude outcomes unavailable by the September model-selection deadline. These are development selection scores, not an independent nested-CV assessment.

In [14]:
tuning = pd.read_csv(R / "history_tuning.csv")
tuning.round(4)

,smoothing_m,long_window,decision_hour,best_C,fold_m5,fold_m6,fold_m7,fold_m8,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_mean_C0.001
0,5,14,20,0.100,0.4350,0.5055,0.5523,0.3933,0.4715,0.0710,0.4690
1,5,28,20,0.001,0.4392,0.4965,0.5402,0.3940,0.4675,0.0641,0.4675
2,5,56,20,0.010,0.4352,0.5034,0.5450,0.3916,0.4688,0.0685,0.4671
3,20,14,20,0.100,0.4310,0.5054,0.5506,0.3924,0.4699,0.0714,0.4675
4,20,28,20,0.001,0.4369,0.4956,0.5393,0.3942,0.4665,0.0639,0.4665
5,20,56,20,0.010,0.4324,0.5024,0.5441,0.3913,0.4676,0.0686,0.4657
6,50,14,20,0.010,0.4238,0.5019,0.5474,0.3896,0.4657,0.0720,0.4637
7,50,28,20,0.001,0.4327,0.4928,0.5372,0.3928,0.4639,0.0639,0.4639
8,50,56,20,0.010,0.4287,0.4996,0.5423,0.3896,0.4650,0.0688,0.4634
9,5,14,0,0.003,0.4070,0.5014,0.5409,0.3863,0.4589,0.0741,0.4583


In [15]:
grid = tuning[tuning.decision_hour == config.DECISION_HOUR].pivot(index="smoothing_m", columns="long_window", values="cv_pr_auc_mean")
grid.round(4)

long_window,14,28,56
smoothing_m,,,
5,0.4715,0.4675,0.4688
20,0.4699,0.4665,0.4676
50,0.4657,0.4639,0.4650


* **Selected pair:** m = 5, long window = 14 days, C = 0.1, mean CV AP **0.4715**, versus 0.4665 for the earlier (20, 28) pair at its best C. The pipeline checks that the configured values agree with the selection table.
* **Joint C tuning retains the same pair:** its fixed-C = 0.001 score is **0.4690**. This agreement does not remove the optimism of selecting and reporting on the same folds.
* The 20:00 clock scores 0.4715, compared with 0.4589 at 00:00 (approximately a two-day lag) and 0.4814 at 24:00. The 24:00 scenario permits outcomes unavailable at the chosen 20:00 deadline. These scenarios move fitting and selection cutoffs as well as feature-history cutoffs, so their score differences do not isolate a feature-leakage effect.

## 7. Should the month be a feature? (rolling CV)

In [16]:
splits = data.assign_split(df)
pd.crosstab(df["month"], splits).loc[:, list(config.SPLITS)]

col_0,train,calib,test
month,,,
1,27004,0,0
2,24951,0,0
3,28834,0,0
4,28330,0,0
5,28796,0,0
6,28243,0,0
7,29425,0,0
8,29327,0,0
9,0,27574,0


In [17]:
pd.read_csv(R / "month_feature_cv.csv").round(4)

,features,model,fold_m5,fold_m6,fold_m7,fold_m8,cv_pr_auc_mean
0,without month,logreg,0.4350,0.5055,0.5523,0.3933,0.4715
1,without month,xgboost,0.4338,0.4193,0.5466,0.3899,0.4474
2,with month,logreg,0.4256,0.5065,0.5492,0.3951,0.4691
3,with month,xgboost,0.3737,0.4468,0.5524,0.3889,0.4405


* Adding cyclic month features lowers mean CV AP for both models: LogReg 0.4715 → 0.4691 and fixed-tree XGBoost 0.4474 → 0.4405. For each model, two folds improve and two worsen. Training contains only months 1–8 in one year; these checks do not establish a reusable seasonal effect. **Decision: no month feature**, supported by development CV. The earlier test-based justification remains withdrawn.

## 8. Why day-level history is a separate group

In [18]:
tr = df.loc[train]
eda.distinct_values(X.loc[train], groups["history_day"] + [f"hist_rate_carrier_origin_dest_{config.LONG_WINDOW}d", f"hist_rate_tailnum_{config.LONG_WINDOW}d"], tr["date"])

,distinct_values,values_per_day,rows,days
hist_rate_network_1d,236,1.0000,211684,236
hist_rate_network_7d,236,1.0000,211684,236
hist_rate_origin_1d,708,3.0000,211684,236
hist_logn_origin_1d,161,2.9873,211684,236
hist_cancel_network_1d,233,1.0000,211684,236
hist_cancel_origin_1d,491,2.9110,211684,236
hist_meandelay_origin_1d,708,3.0000,211684,236
hist_rate_carrier_origin_dest_14d,48999,207.6229,211684,236
hist_rate_tailnum_14d,28407,120.3686,211684,236


`hist_rate_network_1d` has one value per training day, repeated across many flights; per-origin variants have about three. Daily repetition limits the independent temporal variation and creates a possible memorisation risk for flexible models. It does not demonstrate that memorisation occurred.

The group is kept separate for ablation. Adding it increases LogReg mean CV AP by **0.0114**, improving 3/4 folds. For XGBoost, the mean gain is **−0.0007**, despite improvement in 3/4 folds. These are checks at the selected settings.

## 9. History features vs target (training window only)

In [19]:
hist_cols = [c for c in groups["history_entity"] + groups["history_day"] if "rate" in c or "mean" in c]
corr = X.loc[train, hist_cols].corrwith(df.loc[train, "is_delayed"]).sort_values()
fig = plots.hbar(corr.values, corr.index.tolist(), "Correlation with is_delayed (train window)", "Pearson r")
plots.save(fig, "feat_history_corr"); fig

<Figure size 550x444 with 1 Axes>

Every history rate correlates positively with the target. The **finest-grained keys** (specific flight number, origin × hour slot) are the strongest single predictors, and coarse keys are the weakest. Individually the correlations are modest, as expected for a day-ahead problem without weather data.